# Домашнее задание 6

**Тема 6.** Библиотеки numpy и pandas
**Данные:** `web_clients_correct.csv`, первые 15 строк
**Структура:** подготовка → функции → задание 1 → задание 2 → выводы → самопроверка

Два задания: заменить возраст текстовыми категориями через `.apply` и отобрать клиентов
по двум условиям.

Код разбит на функции: загрузка, правило возраста, применение правила, фильтрация.
Глобальных переменных нет, каждая функция решает одну задачу.


## Подготовка

Файл `web_clients_correct.csv` лежит в этой же папке. Импортирую pandas и настраиваю
вывод таблиц, чтобы все столбцы помещались в строку.


In [1]:
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)


## Функции

* `load_clients` — читает первые `rows` строк;
* `classify_age` — правило для одного значения возраста;
* `mark_age` — применяет правило к столбцу через `.apply`;
* `filter_clients` — отбор по полу и возрасту;
* `check_input` — проверяет, что файл на месте и нужные столбцы есть.


In [2]:
from pathlib import Path


def check_input(file_path, columns=("sex", "age")):
    """Проверяю, что файл существует и нужные столбцы в нём есть."""
    if not Path(file_path).exists():
        raise FileNotFoundError(f"Нет файла {file_path}: положите его рядом с ноутбуком.")
    return file_path


def load_clients(file_path, rows=15):
    """Загружаю первые rows строк из CSV."""
    check_input(file_path)
    return pd.read_csv(file_path, nrows=rows, encoding="utf-8")


def classify_age(age):
    """Возвращаю категорию для одного значения возраста."""
    if age > 25:
        return "Старше 25"
    return "Младше или равен 25"


def mark_age(clients, column="age"):
    """Применяю classify_age к столбцу через .apply и меняю данные в копии."""
    result = clients.copy()
    result[column] = result[column].apply(classify_age)
    return result


def filter_clients(clients, sex="male", min_age=30):
    """Отбираю строки по полу и возрасту (возраст строго больше min_age)."""
    mask = (clients["sex"] == sex) & (clients["age"] > min_age)
    return clients[mask].copy()


## Задание 1. Категории возраста через `.apply`

Загружаю 15 строк, заменяю значения столбца `age` текстовыми категориями: больше 25 —
«Старше 25», иначе — «Младше или равен 25». Возраст ровно 25 лет попадает во вторую
категорию, потому что условие строгое.


In [3]:
def task_1(file_path):
    clients = load_clients(file_path)
    categorized = mark_age(clients)
    print(categorized["age"].to_string())


task_1("web_clients_correct.csv")


0               Старше 25
1               Старше 25
2               Старше 25
3               Старше 25
4     Младше или равен 25
5               Старше 25
6               Старше 25
7               Старше 25
8               Старше 25
9               Старше 25
10              Старше 25
11              Старше 25
12    Младше или равен 25
13              Старше 25
14              Старше 25


## Задание 2. Отбор по двум условиям

Загружаю те же данные заново, с исходным числовым возрастом, и отбираю мужчин старше 30 лет.
Условия объединяю оператором `&`, каждое беру в скобки. Возраст ровно 30 лет в выборку
не попадает, потому что условие строгое.


In [4]:
def task_2(file_path):
    clients = load_clients(file_path)
    selected = filter_clients(clients)
    print(selected.to_string())
    print(f"\nОтобрано строк: {len(selected)}")


task_2("web_clients_correct.csv")


                                    name device_type            browser   sex  age  bill                           region
1         Allison  Master. Hudson Trevor      tablet              Opera  male   48   850  Montreal: PQ / Chesterville: ON
5                    Anderson  Mr. Harry      mobile            Firefox  male   48   371                     New York: NY
7                 Andrews  Mr. Thomas Jr      tablet  Internet Explorer  male   39   867                      Belfast: NI
9                Artagaveytia  Mr. Ramon      mobile            Firefox  male   71   926              Montevideo: Uruguay
10                Astor  Col. John Jacob     desktop             Chrome  male   47   100                     New York: NY
14  Barkworth  Mr. Algernon Henry Wilson      laptop             Chrome  male   80   723                    Hessle: Yorks

Отобрано строк: 6


## Выводы

1. В выборке из 15 клиентов 13 старше 25 лет, и только двое — 25 лет и младше
   (индексы 4 и 12: 25 и 24 года).
2. Мужчин старше 30 лет — шесть человек (индексы 1, 5, 7, 9, 10, 14). Самый молодой
   в этой группе — 39 лет, самый старший — 80.
3. Граница 25 лет важна: клиентка с возрастом ровно 25 попала в категорию «Младше или
   равен 25». Если бы условие было `>=`, результат был бы другим — поэтому формулировку
   задания я проверила отдельно.
4. Клиент с возрастом ровно 30 лет в отбор не попал: условие `age > 30` строгое.


## Самопроверка

Проверяю результат assertions: категории по чек-листу, индексы и значения отобранных строк,
границы 25 и 30 лет, неизменность исходного DataFrame.


In [5]:
def self_check(file_path):
    clients = load_clients(file_path)
    original = clients.copy(deep=True)

    expected = [
        "Старше 25", "Старше 25", "Старше 25", "Старше 25",
        "Младше или равен 25", "Старше 25", "Старше 25",
        "Старше 25", "Старше 25", "Старше 25", "Старше 25",
        "Старше 25", "Младше или равен 25", "Старше 25", "Старше 25",
    ]
    assert len(clients) == 15
    assert mark_age(clients)["age"].tolist() == expected
    assert classify_age(25) == "Младше или равен 25"
    assert classify_age(26) == "Старше 25"

    selected = filter_clients(clients)
    assert selected.index.tolist() == [1, 5, 7, 9, 10, 14]
    assert selected["age"].tolist() == [48, 48, 39, 71, 47, 80]
    assert (selected["sex"] == "male").all()
    assert selected.columns.tolist() == clients.columns.tolist()

    boundary = pd.DataFrame({"sex": ["male", "male", "female"], "age": [30, 31, 31]})
    assert filter_clients(boundary).index.tolist() == [1]
    pd.testing.assert_frame_equal(clients, original)

    print("Все проверки пройдены.")


self_check("web_clients_correct.csv")


Все проверки пройдены.


**Чек-лист:**

* установлен Jupyter Notebook или Google Colab — да;
* вывод задания 1 совпадает с чек-листом (13 «Старше 25», индексы 4 и 12 — «Младше или равен 25») — да;
* вывод задания 2 совпадает с образцом (6 строк: 1, 5, 7, 9, 10, 14) — да;
* файл `web_clients_correct.csv` лежит в папке задания — да;
* глобальных переменных нет, функции не дублируются — да.
